# 05 · Blocos empilhados e conexões residuais

**Peça nova:** o `Block` (atenção + MLP) e a **conexão residual**. Um par atenção + MLP dá uma rodada de "comunicar e pensar". Para raciocínios mais longos, empilhamos vários blocos. Só que redes profundas são difíceis de treinar: o gradiente precisa atravessar todas as camadas, e em cada uma ele pode encolher ou explodir.

A conexão residual muda a conta de `x = f(x)` para `x = x + f(x)`. Cada bloco passa a **somar uma correção** a um fluxo que atravessa a rede inteira (o *residual stream*). Na volta, a soma copia o gradiente direto para as camadas de baixo: uma "estrada" sem obstáculos desde a loss até o embedding.

Daqui em diante, `FeedForward` vem de `lab.model`. Ainda **sem normalização** (fica para o 06). Diário: `F1-RES`.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from lab.data import get_batch, shakespeare
from lab.model import FeedForward, LanguageModel, MultiHeadAttention, count_params
from lab.plot import compare, table
from lab.train import pick_device, train

DEVICE = pick_device()
ds = shakespeare()
tok, V = ds.tok, ds.tok.vocab_size
BLOCK, D, STEPS, LR = 32, 64, 3000, 1e-3
print(ds, "| dispositivo:", DEVICE)

## A peça: `Block`, com a residual opcional

In [ ]:
class Block(nn.Module):
    def __init__(self, d_model, n_head, block_size, residual=True):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_head, block_size)
        self.mlp = FeedForward(d_model)
        self.residual = residual

    def forward(self, x):
        if self.residual:
            x = x + self.attn(x)   # soma a correção da atenção
            x = x + self.mlp(x)    # soma a correção da MLP
        else:
            x = self.attn(x)       # substitui
            x = self.mlp(x)
        return x


class DeepLM(LanguageModel):
    def __init__(self, vocab_size, block_size=32, d_model=64, n_head=4, n_layer=4, residual=True):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_head, block_size, residual) for _ in range(n_layer)])
        self.lm_head = nn.Linear(d_model, vocab_size)

    def logits(self, idx):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        return self.lm_head(x)

## O gradiente chega às primeiras camadas?

Antes de treinar, faça um único passo de backpropagation num modelo de 8 blocos e meça o tamanho (norma) do gradiente em cada bloco.

In [ ]:
def grad_norms(model):
    model.to("cpu").train()
    torch.manual_seed(0)
    model.zero_grad()
    _, loss = model(*get_batch(ds.train, 32, BLOCK))
    loss.backward()
    return [torch.cat([p.grad.flatten() for p in b.parameters()]).norm().item() for b in model.blocks]

fig, ax = plt.subplots(figsize=(7, 4))
for residual in [True, False]:
    torch.manual_seed(1337)
    norms = grad_norms(DeepLM(V, BLOCK, D, n_layer=8, residual=residual))
    ax.plot(range(8), norms, "o-", label="com residual" if residual else "sem residual")
ax.set_yscale("log")
ax.set_xlabel("bloco (0 = mais perto da entrada)")
ax.set_ylabel("norma do gradiente")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## Treino de referência: 4 blocos, com e sem residual

**Minha previsão:** _(qual termina com loss menor? a diferença aparece logo no início ou só no fim?)_

In [ ]:
for residual in [True, False]:
    torch.manual_seed(1337)
    name = "05-res-L4" if residual else "05-sem-res-L4"
    run = train(DeepLM(V, BLOCK, D, n_layer=4, residual=residual), ds, name=name, steps=STEPS, lr=LR, device=DEVICE)
    print(run.sample[:200], "\n")

In [ ]:
compare(["04-mlp", "05-res-L4", "05-sem-res-L4"], title="1 bloco vs. 4 blocos com e sem residual")
table(["04-mlp", "05-res-L4", "05-sem-res-L4"])

---
## Experimentos

### 1. Profundidade
Com residual, de 1 a 8 blocos; sem residual, 8 blocos.

**Minha previsão:** _(o ganho cresce sempre com a profundidade? o modelo de 8 blocos sem residual treina?)_

In [ ]:
models = {}
for n_layer in [1, 2, 8]:  # o de 4 blocos já foi treinado acima
    torch.manual_seed(1337)
    models[n_layer] = m = DeepLM(V, BLOCK, D, n_layer=n_layer)
    print(f"{n_layer} blocos: {count_params(m):,} parâmetros")
    train(m, ds, name=f"05-res-L{n_layer}", steps=STEPS, lr=LR, device=DEVICE, verbose=False, sample_tokens=0)
torch.manual_seed(1337)
train(DeepLM(V, BLOCK, D, n_layer=8, residual=False), ds, name="05-sem-res-L8", steps=STEPS, lr=LR, device=DEVICE, verbose=False, sample_tokens=0)
names = [f"05-res-L{n}" for n in [1, 2, 4, 8]] + ["05-sem-res-L8"]
compare(names, title="Profundidade, com e sem residual")
table(names)

### 2. O tamanho do fluxo residual cresce com a profundidade
Com residual, cada bloco **soma** algo ao fluxo. Meça o desvio padrão de `x` na saída de cada bloco do modelo de 8 blocos.

**Minha previsão:** _(o desvio cresce, encolhe ou fica estável de bloco em bloco?)_

In [ ]:
def activation_std(model):
    stds = []
    hooks = [b.register_forward_hook(lambda mod, inp, out: stds.append(out.std().item())) for b in model.blocks]
    model.eval()
    torch.manual_seed(0)
    with torch.no_grad():
        model(get_batch(ds.val, 32, BLOCK, next(model.parameters()).device)[0])
    for h in hooks:
        h.remove()
    return stds

plt.plot(activation_std(models[8]), "o-")  # o modelo de 8 blocos treinado no experimento 1
plt.xlabel("bloco")
plt.ylabel("desvio padrão do fluxo residual")
plt.grid(alpha=0.3)
plt.show()

Esse crescimento é um dos problemas que a normalização (próximo notebook) resolve: cada bloco recebe uma entrada numa escala diferente.

---
## Para o diário (`F1-RES-01`)

- O que o gráfico de gradientes mostrou sem residual? Ligue isso à curva de treino de 8 blocos.
- Profundidade ajudou sempre? Onde o ganho parou?
- Explique com suas palavras por que `x + f(x)` facilita o caminho do gradiente.

**Próximo:** [06 · normalização](06_normalizacao.ipynb).